In [0]:
from pyspark.sql import functions as F
spark.sql("USE CATALOG olist")

In [0]:
dim_customer = (
    spark.table("olist.silver.customers")
    .withColumn("customer_key", F.xxhash64("customer_unique_id"))
    .select("customer_key", "customer_unique_id", "customer_zip_code_prefix",
            "customer_city", "customer_state")
)

dim_customer.write.format("delta").mode("overwrite").saveAsTable("olist.gold.dim_customer")
print(f"dim_customer: {dim_customer.count()} rows")

In [0]:
dim_product = (
    spark.table("olist.silver.products").alias("p")
    .join(spark.table("olist.silver.product_categories").alias("c"),
          on="product_category_name", how="left")
    .withColumn("product_key", F.xxhash64("product_id"))
    .select("product_key", "product_id", "product_category_name",
            "category_name_english", "product_weight_g",
            "product_length_cm", "product_height_cm", "product_width_cm")
)

dim_product.write.format("delta").mode("overwrite").saveAsTable("olist.gold.dim_product")
print(f"dim_product: {dim_product.count()} rows")

In [0]:
dim_seller = (
    spark.table("olist.silver.sellers")
    .withColumn("seller_key", F.xxhash64("seller_id"))
    .select("seller_key", "seller_id", "seller_zip_code_prefix",
            "seller_city", "seller_state")
)

dim_seller.write.format("delta").mode("overwrite").saveAsTable("olist.gold.dim_seller")
print(f"dim_seller: {dim_seller.count()} rows")

In [0]:
date_range = (
    spark.table("olist.silver.orders")
    .select(F.min("order_purchase_timestamp").alias("min_d"),
            F.max("order_purchase_timestamp").alias("max_d"))
    .collect()[0]
)

dim_date = (
    spark.sql(f"""
        SELECT explode(sequence(
            to_date('{date_range.min_d}'), 
            to_date('{date_range.max_d}'), 
            interval 1 day
        )) as date
    """)
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("week", F.weekofyear("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
)

dim_date.write.format("delta").mode("overwrite").saveAsTable("olist.gold.dim_date")
print(f"dim_date: {dim_date.count()} rows")

In [0]:
fact_order_items = (
    spark.table("olist.silver.order_items").alias("oi")
    .join(spark.table("olist.silver.orders").alias("o"), on="order_id", how="inner")
    .join(spark.table("olist.silver.customer_orders").alias("co"), on="customer_id", how="inner")
    .withColumn("customer_key", F.xxhash64("customer_unique_id"))
    .withColumn("product_key", F.xxhash64("oi.product_id"))
    .withColumn("seller_key", F.xxhash64("oi.seller_id"))
    .withColumn("order_date_key", F.date_format("order_purchase_timestamp", "yyyyMMdd").cast("int"))
    .withColumn("item_total", F.col("price") + F.col("freight_value"))
    .select(
        "order_id", "order_item_id",
        "customer_key", "product_key", "seller_key", "order_date_key",
        "order_status",
        "price", "freight_value", "item_total"
    )
    .withColumn("_loaded_at", F.current_timestamp())
    .withColumn("_source", F.lit("silver.order_items + orders + customer_orders"))
)

fact_order_items.write.format("delta").mode("overwrite").saveAsTable("olist.gold.fact_order_items")
print(f"fact_order_items: {fact_order_items.count()} rows")

## Why normalise in silver, denormalise in gold
Silver stays 3NF to guarantee data integrity — each fact lives in exactly 
one place, so cleaning/dedup logic only has to run once per entity. Gold 
deliberately flattens into a star schema so analysts can answer questions 
like "revenue by state" or "top products" with one or two joins instead of 
chaining through five normalised tables — this trades some storage 
redundancy for query simplicity and speed, which is the right trade-off 
for a layer meant to be queried directly by BI tools and stakeholders.